In [10]:
# ==========================================
# CELL 1: Environment Setup & Dataset Check
# ==========================================
!pip install -q timm albumentations opencv-python matplotlib

import os
import glob
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Check GPU Availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Active PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f"[+] GPU Model: {torch.cuda.get_device_name(0)}")

# Check for Kaggle Input Datasets
kaggle_input_dir = "/kaggle/input"
available_datasets = os.listdir(kaggle_input_dir) if os.path.exists(kaggle_input_dir) else []
print(f"[+] Found Kaggle Input Datasets: {available_datasets}")

[+] Active PyTorch Device: cpu
[+] Found Kaggle Input Datasets: []


In [11]:
# =========================================================
# CELL 2: Hybrid CNN-Vision Transformer Architecture
# =========================================================

def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / (window_size * window_size)))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    """Window-based Multi-Head Self-Attention with relative positional bias."""
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
        )

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        self.register_buffer("relative_position_index", relative_coords.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q * self.scale) @ k.transpose(-2, -1)

        bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1
        ).permute(2, 0, 1).contiguous()
        attn = F.softmax(attn + bias.unsqueeze(0), dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj(x)


class SwinTransformerBlock(nn.Module):
    """Swin Transformer Block."""
    def __init__(self, dim, num_heads, window_size=7, shift_size=0):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, (window_size, window_size), num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).contiguous()

        pad_h = (self.window_size - H % self.window_size) % self.window_size
        pad_w = (self.window_size - W % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        
        Hp, Wp = H + pad_h, W + pad_w
        shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2)) if self.shift_size > 0 else x

        x_windows = window_partition(shifted_x, self.window_size).view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(self.norm1(x_windows)).view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, Hp, Wp)

        x_out = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2)) if self.shift_size > 0 else shifted_x
        if pad_h > 0 or pad_w > 0:
            x_out = x_out[:, :H, :W, :].contiguous()

        x = x + x_out
        x = x + self.mlp(self.norm2(x))
        return x.permute(0, 3, 1, 2).contiguous()


class LightweightTransformerNeck(nn.Module):
    def __init__(self, in_channels=[256, 512, 1024], out_channels=256, num_blocks=2):
        super().__init__()
        self.p3_proj = nn.Conv2d(in_channels[0], out_channels, 1)
        self.p4_proj = nn.Conv2d(in_channels[1], out_channels, 1)
        self.p5_proj = nn.Conv2d(in_channels[2], out_channels, 1)

        self.trans_p5 = nn.ModuleList([SwinTransformerBlock(out_channels, 4, 7, 0 if i % 2 == 0 else 3) for i in range(num_blocks)])
        self.trans_p4 = nn.ModuleList([SwinTransformerBlock(out_channels, 4, 7, 0 if i % 2 == 0 else 3) for i in range(num_blocks)])
        self.trans_p3 = nn.ModuleList([SwinTransformerBlock(out_channels, 4, 7, 0 if i % 2 == 0 else 3) for i in range(num_blocks)])

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")

    def forward(self, features):
        P3, P4, P5 = features
        P3_p, P4_p, P5_p = self.p3_proj(P3), self.p4_proj(P4), self.p5_proj(P5)

        for blk in self.trans_p5:
            P5_p = blk(P5_p)
        P4_fused = P4_p + self.upsample(P5_p)
        for blk in self.trans_p4:
            P4_fused = blk(P4_fused)
        P3_fused = P3_p + self.upsample(P4_fused)
        for blk in self.trans_p3:
            P3_fused = blk(P3_fused)

        return [P3_fused, P4_fused, P5_p]


class ConvBlock(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=1):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class CSPDarknetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = ConvBlock(3, 64, 3, 2, 1)
        self.s1 = nn.Sequential(ConvBlock(64, 128, 3, 2, 1), ConvBlock(128, 128))
        self.p3 = nn.Sequential(ConvBlock(128, 256, 3, 2, 1), ConvBlock(256, 256))
        self.p4 = nn.Sequential(ConvBlock(256, 512, 3, 2, 1), ConvBlock(512, 512))
        self.p5 = nn.Sequential(ConvBlock(512, 1024, 3, 2, 1), ConvBlock(1024, 1024))

    def forward(self, x):
        x = self.stem(x)
        x = self.s1(x)
        p3 = self.p3(x)
        p4 = self.p4(p3)
        p5 = self.p5(p4)
        return [p3, p4, p5]


class DecoupledHead(nn.Module):
    def __init__(self, in_channels=256, num_classes=10):
        super().__init__()
        self.cls_conv = nn.Sequential(ConvBlock(in_channels, in_channels), nn.Conv2d(in_channels, num_classes, 1))
        self.reg_conv = nn.Sequential(ConvBlock(in_channels, in_channels), nn.Conv2d(in_channels, 4, 1))

    def forward(self, x):
        return self.cls_conv(x), self.reg_conv(x)


class HybridDetector(nn.Module):
    def __init__(self, num_classes=10, num_transformer_blocks=2):
        super().__init__()
        self.backbone = CSPDarknetBackbone()
        self.neck = LightweightTransformerNeck(num_blocks=num_transformer_blocks)
        self.head_p3 = DecoupledHead(256, num_classes)
        self.head_p4 = DecoupledHead(256, num_classes)
        self.head_p5 = DecoupledHead(256, num_classes)

    def forward(self, x):
        p3, p4, p5 = self.backbone(x)
        n3, n4, n5 = self.neck([p3, p4, p5])
        cls_p3, reg_p3 = self.head_p3(n3)
        cls_p4, reg_p4 = self.head_p4(n4)
        cls_p5, reg_p5 = self.head_p5(n5)
        return {"p3": (cls_p3, reg_p3), "p4": (cls_p4, reg_p4), "p5": (cls_p5, reg_p5)}

In [12]:
# =========================================================
# CELL 3: Dataset Loader Pipeline
# =========================================================

class KaggleTrafficDataset(Dataset):
    def __init__(self, num_samples=200, img_size=640, num_classes=10):
        self.num_samples = num_samples
        self.img_size = img_size
        self.num_classes = num_classes

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        image = torch.randn(3, self.img_size, self.img_size)
        targets = {
            "p3_cls": torch.randint(0, 2, (self.num_classes, 80, 80)).float(),
            "p3_reg": torch.randn(4, 80, 80),
            "p4_cls": torch.randint(0, 2, (self.num_classes, 40, 40)).float(),
            "p4_reg": torch.randn(4, 40, 40),
            "p5_cls": torch.randint(0, 2, (self.num_classes, 20, 20)).float(),
            "p5_reg": torch.randn(4, 20, 20)
        }
        return image, targets

train_dataset = KaggleTrafficDataset(num_samples=160)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
print(f"[+] Loaded {len(train_dataset)} training samples into PyTorch DataLoader")

[+] Loaded 160 training samples into PyTorch DataLoader


In [13]:
# =========================================================
# CELL 4: Training Engine Execution
# =========================================================

def train_hybrid_model(epochs=5, num_blocks=2):
    model = HybridDetector(num_classes=10, num_transformer_blocks=num_blocks).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    bce_loss = nn.BCEWithLogitsLoss()
    l1_loss = nn.SmoothL1Loss()
    scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None

    loss_history = []
    print(f"\n--- Starting Training for {epochs} Epochs ({num_blocks} Transformer Blocks) ---")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        start = time.time()

        for images, targets in train_loader:
            images = images.to(device)
            targets = {k: v.to(device) for k, v in targets.items()}

            optimizer.zero_grad()

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    preds = model(images)
                    loss = 0.0
                    for s in ["p3", "p4", "p5"]:
                        loss += bce_loss(preds[s][0], targets[f"{s}_cls"]) + l1_loss(preds[s][1], targets[f"{s}_reg"])
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                preds = model(images)
                loss = 0.0
                for s in ["p3", "p4", "p5"]:
                    loss += bce_loss(preds[s][0], targets[f"{s}_cls"]) + l1_loss(preds[s][1], targets[f"{s}_reg"])
                loss.backward()
                optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        loss_history.append(avg_loss)
        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_loss:.4f} | Time: {time.time() - start:.2f}s")

    # Plot Loss Curve
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, epochs + 1), loss_history, marker='o', color='b')
    plt.title(f"Hybrid Model Training Loss ({num_blocks} Transformer Blocks)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()

train_hybrid_model(epochs=5, num_blocks=2)


--- Starting Training for 5 Epochs (2 Transformer Blocks) ---


RuntimeError: The size of tensor a (21) must match the size of tensor b (20) at non-singleton dimension 2

In [ ]:
# =========================================================
# CELL 5: Ablation Study (FPS & Latency Benchmark)
# =========================================================

def run_ablation_benchmark():
    dummy_input = torch.randn(1, 3, 640, 640).to(device)
    print("\n" + "=" * 65)
    print(f"{'Config':<22} | {'Params (M)':<12} | {'Latency (ms)':<15} | {'FPS':<10}")
    print("=" * 65)

    for blocks in [1, 2, 4]:
        model = HybridDetector(num_classes=10, num_transformer_blocks=blocks).to(device)
        model.eval()
        params_m = sum(p.numel() for p in model.parameters()) / 1e6

        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = model(dummy_input)
            if device.type == "cuda":
                torch.cuda.synchronize()

        # Measure speed
        t0 = time.time()
        num_runs = 50
        with torch.no_grad():
            for _ in range(num_runs):
                _ = model(dummy_input)
                if device.type == "cuda":
                    torch.cuda.synchronize()

        lat_ms = ((time.time() - t0) / num_runs) * 1000.0
        fps = 1000.0 / lat_ms
        print(f"Hybrid ({blocks} Block{'s' if blocks > 1 else ''})      | {params_m:<12.2f} | {lat_ms:<15.2f} | {fps:<10.1f}")

    print("=" * 65)

run_ablation_benchmark()